# 1/2 — Fine-tune Qwen3-8B (Kaggle, GPU T4 x2)

**Settings:** Accelerator = **GPU T4 x2** (not P100: `--liger` needs compute capability ≥ 7.0), Internet = **On**.

**How to use**
1. *Interactive session:* set `RUN_SMOKE = True` (and optionally `RUN_BASELINE = True`), run all cells up to the smoke test, check it passes.
2. *Long run:* set both flags back to `False`, then **Save Version → Save & Run All (Commit)** (background, ≤ 12 h; ≈ 3 h per epoch).
   Only setup + training run in the commit. The version output (`/kaggle/working/qwen3-8b-lora`) is the trained adapter.
3. *If a commit stops early:* in the next version, **Add Input → Your Work → this notebook's previous version**.
   Cell 3 copies its latest checkpoint into `/kaggle/working` and training resumes from there.
4. Evaluation, ablation and GGUF export run in `kaggle_eval_export.ipynb`.

## 1. Code and dependencies

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # one GPU: QLoRA + Liger must keep all tensors on the same device
%cd /kaggle/working
!rm -rf repo && git clone -q https://github.com/yarenerisen/bist100-agentic-financial-harness.git repo
%cd /kaggle/working/repo
!git log --oneline -1
%cd /kaggle/working/repo/bist_llm
!pip install -q -U "transformers>=4.45" "peft>=0.13" "bitsandbytes>=0.44" accelerate pyarrow liger-kernel
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Settings

In [ ]:
MODEL        = 'Qwen/Qwen3-8B'
ADAPTER      = '/kaggle/working/qwen3-8b-lora'
RESULTS      = '/kaggle/working/results'
EPOCHS       = 1
EVAL_LIMIT   = 40        # MUST match kaggle_eval_export.ipynb: base and fine-tuned use the same first N test rows
RUN_SMOKE    = False     # True in the interactive session only
RUN_BASELINE = False     # True in the interactive session only (base model, full harness, first EVAL_LIMIT rows)

## 3. Resume: copy the latest checkpoint from a previous version (if attached as input)

In [ ]:
import glob, os, shutil, re
prev = sorted(glob.glob('/kaggle/input/*/qwen3-8b-lora/checkpoint-*') + glob.glob('/kaggle/input/*/*/qwen3-8b-lora/checkpoint-*'),
              key=lambda p: int(re.search(r'checkpoint-(\d+)', p).group(1)))
os.makedirs(ADAPTER, exist_ok=True)
if prev:
    dst = os.path.join(ADAPTER, os.path.basename(prev[-1]))
    if not os.path.exists(dst):
        shutil.copytree(prev[-1], dst)
    print('resuming from', prev[-1])
else:
    print('no previous checkpoint attached; training starts from scratch')

## 4. (optional) Baseline: base model with the full harness, same rows as the fine-tuned evaluation

In [ ]:
if RUN_BASELINE:
    !python -m harness.evaluate --backend hf --model {MODEL} --load-4bit --limit {EVAL_LIMIT} --mode E --tag base_E --out-dir {RESULTS}

## 5. (optional) Smoke test: 30 examples, checks memory and the Qwen3 chat format

In [ ]:
if RUN_SMOKE:
    !python training/finetune_qlora.py --model {MODEL} --out /tmp/smoke --limit 30 --epochs 1 --save-steps 1000 --liger

## 6. Training (checkpoint every 20 steps; `--resume` continues from the newest checkpoint in ADAPTER)

In [ ]:
!python training/finetune_qlora.py --model {MODEL} --out {ADAPTER} --epochs {EPOCHS} --liger --resume
import json; r = json.load(open(f'{ADAPTER}/training_report.json'))
print({k: r[k] for k in ('base_model', 'train_examples', 'epochs', 'lr', 'lora_rank', 'minutes')})
print([ (h['step'], round(h['eval_loss'], 4)) for h in r['log_history'] if 'eval_loss' in h])